# AMD/NVDA 原始 SEC 季度展望：受理时间与 Stage1 Qlib 决策日

假设：原始 8-K 的受理日期之后首个 Qlib 交易日 09:46 是保守的条件性使用日期。只审计时间与来源身份，不读取市场收益、拟合或跑账户。SEC 不提供逐份首次公开网页时间，因此受理时间不能冒充首次公开时间。

In [1]:
import os, json, hashlib, datetime
from pathlib import Path
from zoneinfo import ZoneInfo
import pandas as pd
import qlib
from qlib.data import D

root=Path.cwd()
card=root/'research/stage1_amd_nvda_sec_acceptance_clock_audit_20261023'
plan=json.loads((card/'plan.json').read_text())
src=plan['source']
source_root=root/'research/stage1_amd_nvda_sec_guidance_original_probe_20261023'
sha=lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
assert sha(source_root/'quarter_event_adjudication_result.json')==src['quarter_event_result_sha256']
assert sha(source_root/'item202_inventory_result.json')==src['item202_inventory_result_sha256']
assert sha(source_root/'item202_inventory_manifest.json')==src['item202_inventory_manifest_sha256']
assert sha(root/src['Qlib_provider']/'qlib_store_manifest.json')==src['Qlib_provider_manifest_sha256']
selected=json.loads((source_root/'quarter_event_adjudication_result.json').read_text())
inventory=json.loads((source_root/'item202_inventory_result.json').read_text())
manifest=json.loads((source_root/'item202_inventory_manifest.json').read_text())
assert selected['status']=='AMD_NVDA_original_quarter_event_slots_126_identity_complete_one_qualitative_numeric_gap'
assert len(selected['selected_events'])==126 and selected['selected_slots_each_issuer']==63
assert inventory['NAS_archive_sha256']==src['raw_NAS_archive_sha256']==selected['parent_NAS_archive_sha256']
assert inventory['NAS_remote_and_scoped_restore_verified'] is True
checks={x['accession']:x for x in inventory['checks']}
files={x['name']:x for x in manifest['files']}
assert len(checks)==133 and len(files)==266
for x in selected['selected_events']:
    a=x['accession']; c=checks[a]
    assert x['issuer']==c['issuer'] and x['filing_date']==c['filed'] and x['raw_sha256']==c['raw_sha256']
    assert files['item202_'+a+'.txt']['sha256']==x['raw_sha256']
    assert len(c['accepted_header'])==14
assert selected['NVDA_2014_earlier_earnings_without_outlook_accession']=='0001045810-14-000071'
assert next(x for x in selected['selected_events'] if x['accession']=='0001045810-14-000076')['filing_date']=='2014-05-08'

os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(root/src['Qlib_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-01',end_time='2026-10-09',freq='day')).normalize()
assert calendar.is_monotonic_increasing and calendar.is_unique and len(calendar)>3900
ny=ZoneInfo('America/New_York');est=datetime.timezone(datetime.timedelta(hours=-5));utc=datetime.timezone.utc
rows=[]
for x in selected['selected_events']:
    c=checks[x['accession']]
    naive=datetime.datetime.strptime(c['accepted_header'],'%Y%m%d%H%M%S')
    ny_utc=naive.replace(tzinfo=ny).astimezone(utc)
    est_utc=naive.replace(tzinfo=est).astimezone(utc)
    latest=max(ny_utc,est_utc)
    later_day=max(ny_utc.astimezone(ny).date(),est_utc.astimezone(ny).date(),naive.date())
    future=calendar[calendar.date>later_day]
    row={'issuer':x['issuer'],'announcement_year':x['announcement_year'],'slot':x['slot'],'accession':x['accession'],'raw_sha256':x['raw_sha256'],'revenue_outlook':x['revenue_outlook'],'SGML_acceptance_header':c['accepted_header'],'SGML_NY_UTC':ny_utc.isoformat(),'SGML_fixed_EST_UTC':est_utc.isoformat(),'status':'out_of_snapshot'}
    if len(future):
        day=future[0].date()
        decision=datetime.datetime.combine(day,datetime.time(9,46),tzinfo=ny).astimezone(utc)
        margin=(decision-latest).total_seconds()/3600
        lag=(day-later_day).days
        row.update(next_Qlib_session=str(day),decision_UTC=decision.isoformat(),minimum_acceptance_to_decision_hours=margin,calendar_days_after_acceptance=lag,status='conditional_date_gate_pass' if margin>=12 and lag<=7 else 'unknown_timing_margin')
    rows.append(row)
assert len(rows)==126 and len({(x['issuer'],x['announcement_year'],x['slot']) for x in rows})==126
counts={k:sum(x['status']==k for x in rows) for k in sorted({x['status'] for x in rows})}
result={'status':'conservative_AMD_NVDA_SEC_acceptance_to_next_Qlib_session_date_audited','plan_sha256':sha(card/'plan.json'),'parent_quarter_result_sha256':src['quarter_event_result_sha256'],'Qlib_provider_manifest_sha256':src['Qlib_provider_manifest_sha256'],'selected_events':126,'status_counts':counts,'minimum_margin_hours':min(x['minimum_acceptance_to_decision_hours'] for x in rows if 'minimum_acceptance_to_decision_hours' in x),'maximum_lag_calendar_days':max(x['calendar_days_after_acceptance'] for x in rows if 'calendar_days_after_acceptance' in x),'rows':rows,'SEC_exact_first_public_timestamp_available':False,'source_semantic_numeric_harmonization_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2,ensure_ascii=False)+'\n')
summary={'experiment':'stage1_amd_nvda_sec_acceptance_clock_audit_20261023','decision':'conditional_date_gate_only_no_feature_or_account','result_sha256':sha(card/'result.json'),'selected_events':126,'status_counts':counts,'minimum_margin_hours':result['minimum_margin_hours'],'maximum_lag_calendar_days':result['maximum_lag_calendar_days'],'first_public_exact_qualified':False,'numeric_harmonization_qualified':False,'new_external_GETs':0,'Qlib_D_calendar_calls':1,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'objective_achieved':False,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n')
print(json.dumps({k:result[k] for k in ['status','selected_events','status_counts','minimum_margin_hours','maximum_lag_calendar_days']},ensure_ascii=False))

[2921:MainThread](2026-10-23 06:14:51,962) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[2921:MainThread](2026-10-23 06:14:52,348) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[2921:MainThread](2026-10-23 06:14:52,348) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{"status": "conservative_AMD_NVDA_SEC_acceptance_to_next_Qlib_session_date_audited", "selected_events": 126, "status_counts": {"conditional_date_gate_pass": 126}, "minimum_margin_hours": 12.145833333333334, "maximum_lag_calendar_days": 4}


## 中文结论

126/126 个 AMD/NVDA 季度槽位通过保守的下一 Qlib 交易日 09:46 日期门；最短受理至决策间隔 12.1458 小时，最长跨 4 个日历日。只调用一次 `D.calendar`，0 外部请求、市场收益读取、拟合或账户。SEC 没有逐份首次公开时间戳，日期通过不等于完整来源资格；数值、币种和会计范围尚待核验，也没有 Stage1 特征或 SOXX 超额结论。逐项见 `result.json`，决策见 `conclusion.md`。

后续原始申报日修复见 `filing_date_repair_result.json`：AMD 2011-01-21 的定性未知事件映射从 2011-01-21 延后到 2011-01-24，其余 125 个不变，修复后的最短受理至决策间隔为 15.5758 小时。